In [16]:
import os

from pyspark.sql import SparkSession
import pyspark.sql.functions as F 
from pyspark.sql.window import Window

In [17]:
spark = (
    SparkSession.builder
    .appName("pg_spark__Sainov_A")
    .config(
        "spark.jars.packages",
        "org.postgresql:postgresql:42.5.0"
    )
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Spark version: 4.2.0
Spark UI: http://b51f9c1997db:4041


In [18]:
env_names = [
    "POSTGRES_HOST",
    "POSTGRES_PORT",
    "POSTGRES_DB",
    "POSTGRES_USER",
    "POSTGRES_PASSWORD",
]

for name in env_names:
    value = os.getenv(name)

    if "PASSWORD" in name:
        print(name, "=", "найден" if value else "не найден")
    else:
        print(name, "=", value)

POSTGRES_HOST = None
POSTGRES_PORT = None
POSTGRES_DB = None
POSTGRES_USER = postgres
POSTGRES_PASSWORD = найден


In [19]:
import os

PG_HOST = "postgres_source"
PG_PORT = "5432"
PG_DATABASE = "source"
PG_USER = os.getenv("POSTGRES_USER")
PG_PASSWORD = os.getenv("POSTGRES_PASSWORD")

jdbc_url = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DATABASE}"

print("JDBC URL:", jdbc_url)
print("Пользователь:", PG_USER)
print("Пароль найден:", PG_PASSWORD is not None)

JDBC URL: jdbc:postgresql://postgres_source:5432/source
Пользователь: postgres
Пароль найден: True


In [20]:
shops_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "public.shops")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

shops_df.show(truncate=False)
shops_df.printSchema()

+-----+-----------+
|st_id|shop_name  |
+-----+-----------+
|842  |Lenta      |
|843  |Magnit     |
|844  |Spar       |
|845  |Pyaterochka|
|846  |Lenta      |
|847  |Diksi      |
|848  |Lenta      |
|849  |FixPrice   |
|850  |Magnit     |
|851  |Lenta      |
+-----+-----------+

root
 |-- st_id: integer (nullable = true)
 |-- shop_name: string (nullable = true)



In [21]:
shop_timezone_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "public.shop_timezone")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

shop_timezone_df.show(truncate=False)
shop_timezone_df.printSchema()

+-----+---------+
|plant|time_zone|
+-----+---------+
|842  |         |
|842  |RUS07    |
|843  |RUS04    |
|844  |         |
|845  |         |
|845  |RUS05    |
|847  |RUS03    |
|848  |RUS08    |
|848  |         |
|P847 |         |
|E103 |RUS08    |
|-134 |RUS04    |
|0    |         |
|0    |RUS08    |
|848  |         |
+-----+---------+

root
 |-- plant: string (nullable = true)
 |-- time_zone: string (nullable = true)



In [24]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


timezone_prepared_df = (
    shop_timezone_df
    .withColumn(
        "st_id",
        F.expr("try_cast(plant as int)")
    )
    .withColumn(
        "time_zone_clean",
        F.when(
            F.trim(F.col("time_zone")) != "",
            F.trim(F.col("time_zone"))
        )
    )
)

timezone_prepared_df.show(truncate=False)

+-----+---------+-----+---------------+
|plant|time_zone|st_id|time_zone_clean|
+-----+---------+-----+---------------+
|842  |         |842  |NULL           |
|842  |RUS07    |842  |RUS07          |
|843  |RUS04    |843  |RUS04          |
|844  |         |844  |NULL           |
|845  |         |845  |NULL           |
|845  |RUS05    |845  |RUS05          |
|847  |RUS03    |847  |RUS03          |
|848  |RUS08    |848  |RUS08          |
|848  |         |848  |NULL           |
|P847 |         |NULL |NULL           |
|E103 |RUS08    |NULL |RUS08          |
|-134 |RUS04    |-134 |RUS04          |
|0    |         |0    |NULL           |
|0    |RUS08    |0    |RUS08          |
|848  |         |848  |NULL           |
+-----+---------+-----+---------------+



In [25]:
timezone_window = (
    Window
    .partitionBy("st_id")
    .orderBy(
        F.col("time_zone_clean").isNotNull().desc()
    )
)

In [26]:
timezone_unique_df = (
    timezone_prepared_df
    .filter(F.col("st_id").isNotNull())
    .withColumn(
        "row_num",
        F.row_number().over(timezone_window)
    )
    .filter(F.col("row_num") == 1)
    .select(
        "st_id",
        "time_zone_clean"
    )
)

In [27]:
final_df = (
    shops_df
    .join(
        timezone_unique_df,
        on="st_id",
        how="inner"
    )
    .select(
        F.col("st_id"),
        F.col("shop_name"),
        F.coalesce(
            F.regexp_extract(
                F.col("time_zone_clean"),
                r"(\d+)$",
                1
            ).cast("integer"),
            F.lit(3)
        ).alias("tz_code")
    )
    .orderBy("st_id")
)

final_df.show(truncate=False)

+-----+-----------+-------+
|st_id|shop_name  |tz_code|
+-----+-----------+-------+
|842  |Lenta      |7      |
|843  |Magnit     |4      |
|844  |Spar       |3      |
|845  |Pyaterochka|5      |
|847  |Diksi      |3      |
|848  |Lenta      |8      |
+-----+-----------+-------+

